# Newton fractals and basins of attraction

Starting code for the practical task of the *Root finding and optimization* chapter:
<https://dse.iskh.me/solvers>

The notebook has two parts. The first is the simpler example to study — the Newton
fractal of $z^3-1=0$ in the complex plane. The second collects everything needed to
run the two-dimensional problem of the chapter, with and without the line search.

**The task**: draw the equivalent of the first picture for the second problem — the
basins of attraction of the four local maxima and the other critical points of
$F(x,y)$.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [9, 6]

## The simpler example: the Newton fractal of $z^3-1=0$

$z^3-1=0$ has one real root $z=1$ and two more in the complex plane. Run Newton from
every point of a grid in $\mathbb{C}$, and color each starting point by the root it
ends up at, shaded by the number of iterations it took to get there.

The whole grid is iterated at once, as one array — there is no loop over starting
points, only over iterations.

In [ ]:
n, bound, maxiter = 500, 1.5, 30
re,im = np.meshgrid(np.linspace(-bound,bound,n),np.linspace(-bound,bound,n))
z = re + 1j*im                           # grid of starting points in the complex plane
roots = np.exp(2j*np.pi*np.arange(3)/3)  # the three roots of z**3 = 1
iters = np.zeros(z.shape)
for i in range(maxiter):
    z = z - (z**3-1)/(3*z**2)  # Newton step for all starting points at once
    done = (np.abs(z[...,None]-roots).min(axis=-1) < 1e-8) & (iters==0)
    iters[done] = i+1
which = np.abs(z[...,None]-roots).argmin(axis=-1)  # which root was reached

fig, ax = plt.subplots(figsize=(7,7))
ax.imshow(which + 0.7*(1-iters/iters.max()),extent=[-bound,bound,-bound,bound],
          cmap='turbo',origin='lower')
ax.scatter(roots.real,roots.imag,c='white',edgecolors='black',zorder=3)
ax.set_aspect('equal')
plt.show()

The three basins are not three tidy wedges: every point on a boundary has all
three basins arbitrarily close to it, so an arbitrarily small change in the starting
value sends the solver to a different root.

## The two-dimensional problem

A sum of four bumps: three Gaussian hills and one curved ridge. Each term is written
as $a_i\exp(-q_i)$, so that one chain rule serves all four:

$$\nabla F = -\sum_i g_i \nabla q_i, \qquad
\nabla^2 F = \sum_i g_i\big[(\nabla q_i)(\nabla q_i)^\top - \nabla^2 q_i\big]$$

In [ ]:
def quad(x,y,c,d,A,B):
    '''Exponent q of a plain Gaussian bump, with its gradient and Hessian'''
    q   = (x-c)**2/A + (y-d)**2/B
    dq  = [2*(x-c)/A, 2*(y-d)/B]
    d2q = [[2/A+0*x, 0*x],[0*x, 2/B+0*x]]
    return q,dq,d2q

def quad_banana(x,y):
    '''Exponent q of the curved ridge, with its gradient and Hessian'''
    u = x - 0.55
    v = y - 0.63 + 2.5*u**2
    q   = u**2/0.055 + v**2/0.0025
    dq  = [2*u/0.055 + 4000*u*v, 800*v]
    d2q = [[2/0.055 + 4000*v + 20000*u**2, 4000*u],[4000*u, 800+0*x]]
    return q,dq,d2q

def bumps(x,y):
    '''The four terms of F: amplitude a, exponent q, and the derivatives of q'''
    return [(1.00,) + quad(x,y,0.22,0.27,0.018,0.030),
            (0.94,) + quad(x,y,0.73,0.25,0.080,0.012),
            (0.78,) + quad_banana(x,y),
            (1.20,) + quad(x,y,0.55,0.48,0.005,0.010)]

def F(x,y):
    return sum(a*np.exp(-q) for a,q,dq,d2q in bumps(x,y))

In [ ]:
def G(x,y):
    '''Gradient of F: grad(a exp(-q)) = -a exp(-q) grad(q)'''
    out = [0,0]
    for a,q,dq,d2q in bumps(x,y):
        g = a*np.exp(-q)
        for k in range(2):
            out[k] = out[k] - g*dq[k]
    return out

def H(x,y):
    '''Hessian of F: H(a exp(-q)) = a exp(-q) [grad(q) grad(q)' - H(q)]'''
    out = [[0,0],[0,0]]
    for a,q,dq,d2q in bumps(x,y):
        g = a*np.exp(-q)
        for k in range(2):
            for j in range(2):
                out[k][j] = out[k][j] + g*(dq[k]*dq[j] - d2q[k][j])
    return out

## Contours

In [ ]:
plt.rcParams['figure.figsize'] = [9, 6]
def contour_plot(fun,levels=30,xlim=(0,1.2),ylim=(0,0.72),npoints=200,ax=None,clip=None):
    '''Contour plot of a function of two variables.
    With clip=p the levels are symmetric around zero and cut at the p-th percentile
    of |Z|, which keeps a few extreme values from swamping the picture.
    '''
    X,Y = np.meshgrid(np.linspace(*xlim,npoints),np.linspace(*ylim,npoints))
    Z = fun(X,Y)
    if clip is None:
        lv = np.linspace(Z.min(),Z.max(),levels)
        lv = np.concatenate([np.exp(np.linspace(np.log(Z.min()),np.log(0.2),levels//2)),np.linspace(0.3,Z.max(),levels//2)])
    else:
        c = np.percentile(np.abs(Z),clip)
        lv = np.linspace(-c,c,levels)
    if ax is None:
        fig, ax = plt.subplots()
    ax.contour(X,Y,Z,levels=lv)
    ax.set_aspect('equal','box')
    ax.set_xlim(*xlim)  # fix the window: paths drawn on top may leave it
    ax.set_ylim(*ylim)
    return ax

contour_plot(F)
plt.show()

## Newton's method in two dimensions

The division of the scalar method becomes a linear solve, and the error is measured
with a vector norm.

In [ ]:
def newton2(fun,grad,x0,tol=1e-6,maxiter=100,callback=None):
    '''Newton method for solving a system of equations fun(x)=0,
    where x is a vector of 2 elements and grad is the Jacobian.
    Callback function is invoked at each iteration if given.
    '''
    # conversion to array function of array argument
    npfun  = lambda x: np.asarray(fun(x[0],x[1]))
    npgrad = lambda x: np.asarray(grad(x[0],x[1]))
    x0 = np.asarray(x0,dtype=float)
    for i in range(maxiter):
        x1 = x0 - np.linalg.solve(npgrad(x0),npfun(x0))  # matrix version
        err = np.amax(np.abs(x1-x0))  # vector sup norm
        if callback != None: callback(iter=i,err=err,x0=x0,x1=x1)
        if err<tol: break
        x0 = x1
    else:
        raise RuntimeError('Failed to converge in %d iterations'%maxiter)
    return x1

## The same method with a step-halving line search

The full Newton step is halved until the criterion $F$ increases, so the iterations
always climb. It fails — `No ascent direction` — where the Newton direction does not
point uphill, which happens wherever the Hessian is not negative definite.

In [ ]:
def newton2_ascent(fun,grad,x0,obj=F,tol=1e-6,maxiter=100,maxhalve=25,callback=None):
    '''Newton method with a step-halving line search on the criterion obj.
    A step is accepted only when obj increases, so the iterations always climb.
    '''
    npfun  = lambda x: np.asarray(fun(x[0],x[1]))
    npgrad = lambda x: np.asarray(grad(x[0],x[1]))
    x0 = np.asarray(x0,dtype=float)
    obj0 = obj(*x0)  # criterion at the current point
    for i in range(maxiter):
        step = np.linalg.solve(npgrad(x0),npfun(x0))  # the full Newton step
        lam = 1.0
        for j in range(maxhalve):  # step-halving line search
            x1 = x0 - lam*step
            obj1 = obj(*x1)
            if obj1 > obj0: break  # uphill, accept this lambda
            lam = lam/2
        else:
            raise RuntimeError('No ascent direction at iteration %d'%i)
        err = np.amax(np.abs(x1-x0))
        if callback != None: callback(iter=i,err=err,x0=x0,x1=x1,lam=lam)
        if err<tol: break
        x0, obj0 = x1, obj1
    else:
        raise RuntimeError('Failed to converge in %d iterations'%maxiter)
    return x1

## Running both from the same starting points

In [ ]:
def newton_trace(x0,solver=newton2,**kwargs):
    '''Newton path from x0, with the reason the solver stopped'''
    path = [np.asarray(x0,dtype=float)]
    try:
        solver(G,H,x0,callback=lambda **kw: path.append(kw['x1']),**kwargs)
        status = 'converged'
    except np.linalg.LinAlgError:
        status = 'singular Hessian'
    except RuntimeError as e:
        status = 'no ascent direction' if 'ascent' in str(e) else 'maxiter reached'
    return np.array(path), status

center, side = np.array([0.6,0.43]), 0.05  # the small square of starting points
starts = center + np.random.default_rng(44).uniform(-side/2,side/2,size=(10,2))
colors = plt.cm.autumn(np.linspace(0,0.85,len(starts)))  # slightly different colors

def plot_newton_paths(solver=newton2,starts=starts,colors=colors,**kwargs):
    '''Follow all the starting points at once, each path in its own color'''
    ax = contour_plot(F)
    for c,x0 in zip(colors,starts):
        path, status = newton_trace(x0,solver=solver,**kwargs)
        ax.plot(path[:,0],path[:,1],c=c,marker='.',lw=1.2,zorder=2)  # the path
        ax.scatter(*path[0],c=[c],marker='o',s=25,zorder=3)          # starting point
        if status == 'converged':
            ax.scatter(*path[-1],c=[c],marker='*',s=140,edgecolors='k',lw=.4,zorder=4)
        else:  # mark where the solver gave up
            ax.scatter(*path[-1],c=[c],marker='X',s=80,edgecolors='k',lw=.4,zorder=4)
    ax.scatter([],[],c='grey',marker='o',s=25,label='start')            # legend only
    ax.scatter([],[],c='grey',marker='*',s=140,edgecolors='k',lw=.4,label='converged')
    ax.scatter([],[],c='grey',marker='X',s=80,edgecolors='k',lw=.4,label='gave up')
    ax.legend(loc='upper left',fontsize=8,framealpha=0.8)
    plt.show()

plot_newton_paths()

In [ ]:
plot_newton_paths(newton2_ascent)

## The task

Replace the ten random starting points by a grid covering the whole picture, run the
solver from every point of it, and color the grid by the critical point it reaches —
the two-dimensional counterpart of the fractal above.

Some things to work out along the way.

- **Which critical point is which.** The solver returns coordinates, not labels.
  Collect the endpoints, round them, and keep a list of the distinct ones; then each
  run can be matched to an index in that list.
- **Runs that do not converge.** They need a color of their own — `newton_trace`
  already reports why a run stopped.
- **Maxima against the rest.** The eigenvalues of `H` at the endpoint say whether it
  is a maximum, a minimum or a saddle point: use
  `np.linalg.eigvalsh(np.asarray(H(*xs)))`.
- **Both solvers.** Draw the map twice, with `newton2` and with `newton2_ascent`, and
  compare. The second should have no basins belonging to minima at all.
- **Cost.** One solve per grid point, so a 100x100 grid is 10000 solves. Start with a
  coarse grid, and refine it once the picture looks right.

The fractal above is vectorized over the whole grid at once; here it is simpler to
loop over the starting points, since each run stops at a different iteration.

In [ ]:
# your code here